In [ ]:
import pandas as pd

# URL del archivo CSV hospedado en GitHub (raw)
url = "https://raw.githubusercontent.com/github/innovationgraph/refs/heads/main/data/repositories.csv"

# Carga directa desde la URL
df = pd.read_csv(url)

# Muestra las primeras filas del DataFrame
df

## Rename columns

In [ ]:
df.rename(columns={"repositories": "num_repos"}, inplace=True)
df

## Select South America

In [ ]:
df[ (df["iso2_code"] == "AR") | (df["iso2_code"] == "PE") ]

In [ ]:
# List of South American ISO2 codes
south_america_codes = [
    "AR", "BO", "BR", "CL", "CO", "EC",
    "GY", "PY", "PE", "SR", "UY", "VE", "GF"  # GF optional (French Guiana)
]

# Filter DataFrame
df_south_america = df[df["iso2_code"].isin(south_america_codes)]
df_south_america

In [ ]:
df_south_america.shape

## Sort Dataset

In [ ]:
df[ ["quarter", "year"] ]

In [ ]:
df

In [ ]:
df_south_america.columns

In [ ]:
# Reorder columns
df_south_america = df_south_america[ ["iso2_code", "year", "quarter", "num_repos"] ]
df_south_america

In [ ]:
df_south_america = df_south_america.sort_values(
    by=["iso2_code", "year", "quarter"],
    ascending=[True, True, True]
)
df_south_america

In [ ]:
df_south_america.reset_index(inplace=True)
df_south_america

In [ ]:
df_south_america.drop("index", axis=1, inplace=True)
df_south_america

In [ ]:
df_south_america.shape

In [ ]:
df_south_america["iso2_code"].nunique()

In [ ]:
df_south_america.iso2_code.nunique()

In [ ]:
df_south_america.year.nunique()

In [ ]:
df_south_america.quarter.nunique()

In [ ]:
df_south_america["year"].astype(int)

In [ ]:
lm_df =  df_south_america[["iso2_code", "year", "quarter", "num_repos"]].copy()
lm_df

In [ ]:
import pandas as pd

# Ensure correct order and types
df_south_america = df_south_america[["iso2_code", "year", "quarter", "num_repos"]].copy()
df_south_america["year"] = df_south_america["year"].astype(int)
df_south_america["quarter"] = df_south_america["quarter"].astype(int)

# Define the full cartesian product (all combos)
all_iso2   = sorted(df_south_america["iso2_code"].unique())
all_years  = sorted(df_south_america["year"].unique())
all_quarts = [1, 2, 3, 4]   # or: sorted(df_south_america["quarter"].unique())

In [ ]:
all_years

In [ ]:
full_index = pd.MultiIndex.from_product(
    [all_iso2, all_years, all_quarts],
    names=["iso2_code", "year", "quarter"]
)
full_index

In [ ]:
df_south_america.set_index(["iso2_code", "year", "quarter"]).reindex(full_index).reset_index()

In [ ]:
df_south_america.set_index(["iso2_code", "year", "quarter"]).reindex(full_index)

In [ ]:
# Reindex to the full panel
balanced = (
    df_south_america
    .set_index(["iso2_code", "year", "quarter"])
    .reindex(full_index)
    .reset_index()
)
balanced

In [ ]:
balanced[balanced.num_repos.isna()]

In [ ]:
# Optional: fill missing outcomes (choose ONE)
# If "missing means zero"
balanced["num_repos"] = balanced["num_repos"].fillna(0).astype(int)
balanced

In [ ]:
balanced.to_csv("../_data/github_repos_data_2020_2015_sa.csv")

In [ ]:
type(balanced)

In [ ]:
pip install python-docx

In [ ]:
from docx import Document

# Create a new Word document
doc = Document()
doc.add_heading('Balanced GitHub Repos Data (South America) 2020 -2025', level=1)

# Add a table with the DataFrame content
table = doc.add_table(rows=1, cols=len(balanced.columns))

# Add header row
hdr_cells = table.rows[0].cells
for i, col_name in enumerate(balanced.columns):
    hdr_cells[i].text = str(col_name)

# Add data rows
for row in balanced.itertuples(index=False):
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        row_cells[i].text = str(value)

# Save the document
doc.save("../_data/github_repos_data_2020_2015_sa.docx")